**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**03 | Participant-Grouped Classical and Deep Unimodal Baselines**

![Participant-Grouped Classical and Deep Unimodal Baselines workflow](../../images/03_unimodal_grouped_baselines.svg)

**Research objective:** Compare source-aligned sEMG-only and acceleration-only classifiers on identical previously unseen Development participants.

**Dataset:** Paired E1 windows and person-independent assignments from Notebook 02.

**Notebook scope:** Run a training-prior comparator, Logistic Regression, Random Forest, 1D-CNN, TCN and BiGRU without fitting or selecting on Test people.

**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ---------------------------- 2. Standard Library ----------------------------
import time

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# -------------------- 5. Deep Learning and Compute Device --------------------
import torch
from core.computer_device import computer_device
device = computer_device(min_vram_gb=None)

# ------------------------ 6. Study Research Utilities ------------------------
from ninapro_dataset import ensure_numerical_cache
from ninapro_evaluation import (
    build_classical, classical_features, aligned_probabilities,
    participant_metrics, select_model, predictions_from_probabilities,
)
from ninapro_models import UNIMODAL_MODELS
from ninapro_training import fit_neural, predict_neural

# ---------------------- 7. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
raw_dir = root / "data" / "raw" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


Local changes kept; continuing with the current checkout. Conflicting Git update deferred.
Compute device and system information
OS: Linux 6.6.159+
CPU: x86_64
Installed RAM: 12.7 GiB
NVIDIA GPU: Tesla T4
Total VRAM: 15.00 GiB
PyTorch version: 2.14.1+cu130
CUDA device: Tesla T4
CUDA runtime: 13.0
cuDNN version: 92800
PyTorch total VRAM: 14.56 GiB
Selected compute device: cuda
Research output directory: /content/human-movement-intelligence/multimodal_movement_learning/output


**02. Read Fixed Classes and Paired Development Windows**

Notebook 02 supplies unique window rows, source labels, preserved person roles and fixed Development folds. The reserved Test people are excluded from every model-development operation. In a fresh Colab kernel, the numerical source cache can be reconstructed from original participant archives without changing the canonical window index.

In [3]:
# =========== 02. Read Fixed Classes and Paired Development Windows ===========

NOTEBOOK_NUMBER = 3
index = pd.read_csv(output_dir / "02_window_index.csv")
roles = pd.read_csv(output_dir / "02_participant_split_assignments.csv")
folds = pd.read_csv(output_dir / "02_development_fold_assignments.csv")
classes = np.sort(pd.read_csv(output_dir / "01_label_dictionary.csv").label.to_numpy(int))
role_map = dict(zip(roles.subject_id, roles.role))
fold_map = dict(zip(folds.subject_id, folds.validation_fold))
index["role"] = index.subject_id.map(role_map)
index["validation_fold"] = index.subject_id.map(fold_map)
if index.role.isna().any() or index.loc[index.role == "Development", "validation_fold"].isna().any():
    raise ValueError("Incomplete Notebook 02 person/fold assignments.")

dev = index.loc[index.role == "Development"].copy()
assert not set(dev.subject_id) & set(roles.loc[roles.role == "Test", "subject_id"])
print("Development people:", dev.subject_id.nunique(), "| windows:", len(dev))
print("Independent Development folds:", sorted(dev.validation_fold.unique()))


Development people: 35 | windows: 35000
Independent Development folds: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


**03. Recover Aligned Statistical and Temporal Inputs**

Statistical features use only the original two sensor modalities and are already aligned by the immutable `window_row`. Neural sequences are generated lazily from the same original source-sample intervals. Missing local numerical caches are reconstructed from the official source rather than selecting a different cohort or historical output.

In [4]:
# ============ 03. Recover Aligned Statistical and Temporal Inputs ============

emg_features, acc_features = ensure_numerical_cache(root, index, allow_source_download=True)
print("Feature array shapes:", emg_features.shape, acc_features.shape)
print("Fixed task classes:", classes.tolist())


Feature array shapes: (40000, 48) (40000, 96)
Fixed task classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


**04. Define Matching Classical Model Families**

The majority-prior classifier is a source-independent sanity benchmark. Each sensing modality receives the same fixed Logistic Regression and Random Forest configurations. Within each fold, learned scalers and class weights are fitted on Training participants only; probabilities retain one fixed original-class column order.

In [5]:
# =============== 04. Define Matching Classical Model Families ================

CLASSICAL_IDS = ("B0_prior", "B1_logreg", "B1_rf", "B2_logreg", "B2_rf")
SEED = 42
common_folds = sorted(dev.validation_fold.dropna().unique().astype(int).tolist())
probabilities = {name: np.full((len(index), len(classes)), np.nan, np.float32)
                 for name in (*CLASSICAL_IDS, *UNIMODAL_MODELS)}
classical_metrics, classical_predictions, fit_record = [], [], []
for fold in common_folds:
    fit_rows = dev.loc[dev.validation_fold != fold].copy()
    val_rows = dev.loc[dev.validation_fold == fold].copy()
    if set(fit_rows.subject_id) & set(val_rows.subject_id):
        raise ValueError("A participant appears in both Training and outer Validation.")
    for name in CLASSICAL_IDS:
        started = time.perf_counter()
        model = build_classical(name, seed=SEED)
        model.fit(classical_features(name, emg_features[fit_rows.window_row],
                                     acc_features[fit_rows.window_row]),
                  fit_rows.label.to_numpy(int))
        proba = aligned_probabilities(
            model, classical_features(name, emg_features[val_rows.window_row],
                                      acc_features[val_rows.window_row]), classes,
        )
        seconds = time.perf_counter() - started
        probabilities[name][val_rows.window_row.to_numpy(int)] = proba
        metric = participant_metrics(val_rows.reset_index(drop=True), proba, classes,
                                     model_id=name, fold=fold, seed=SEED)
        metric["fit_and_prediction_seconds"] = seconds
        classical_metrics.append(metric)
        classical_predictions.append(pd.DataFrame({
            "window_key": val_rows.window_key.to_numpy(),
            "subject_id": val_rows.subject_id.to_numpy(),
            "validation_fold": fold, "model_id": name,
            "true_label": val_rows.label.to_numpy(int),
            "predicted_label": predictions_from_probabilities(proba, classes),
        }))
    print(f"Classical grouped fold {fold} complete")
classical_metrics = pd.concat(classical_metrics, ignore_index=True)
pd.concat(classical_predictions, ignore_index=True).to_csv(
    output_dir / "03_oof_predictions.csv", index=False)
classical_metrics.to_csv(output_dir / "03_dev_unimodal_metrics.csv", index=False)
display(classical_metrics.groupby("model_id", as_index=False).macro_f1.mean()
        .sort_values("macro_f1", ascending=False))


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 0 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 1 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 2 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 3 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 4 complete


,model_id,macro_f1
2,B1_rf,0.234836
1,B1_logreg,0.208411
4,B2_rf,0.128978
3,B2_logreg,0.103237
0,B0_prior,0.034971


**05. Define the Shared Neural Training Procedure**

Six separate experiment cells below each fit **one** named architecture on the common Development folds. This shared helper enforces identical hyperparameters, participant-level separation, training-only normalisation, inner early stopping and fixed-class scoring. Epoch logs and per-fold saving make long-running training observable. The datasets and model-selection criterion remain unchanged.

In [ ]:
# =================== 05. Shared Neural Training Procedure ====================

NEURAL_EPOCHS = 10
NEURAL_PATIENCE = 3
NEURAL_BATCH_SIZE = 96

def run_unimodal_arm(model_id):
    """Fit one named architecture across all participant-disjoint folds.

    Report epoch progress, write the model's current fold results and return
    a concise participant-balanced Development summary.
    """
    if model_id not in UNIMODAL_MODELS:
        raise ValueError(f"Unknown unimodal architecture: {model_id}")

    model_scores, model_predictions, model_histories = [], [], []
    development_rows = dev.window_row.to_numpy(int)
    probabilities[model_id][:] = np.nan

    print(f"\nModel: {model_id} | {len(common_folds)} participant-disjoint folds",
          flush=True)
    for fold in common_folds:
        fitting = dev.loc[dev.validation_fold != fold].copy()
        validation = dev.loc[dev.validation_fold == fold].copy()
        if set(fitting.subject_id) & set(validation.subject_id):
            raise ValueError("Overlapping participants in a Development fold.")

        print(
            f"\n{model_id} | fold {fold} | training people "
            f"{fitting.subject_id.nunique()} | validation people "
            f"{validation.subject_id.nunique()}",
            flush=True,
        )
        model, stats, history, info = fit_neural(
            model_id, fitting, processed_dir, classes,
            device=device, seed=SEED, epochs=NEURAL_EPOCHS,
            patience=NEURAL_PATIENCE, batch_size=NEURAL_BATCH_SIZE,
            show_progress=True,
        )
        proba = predict_neural(
            model_id, model, validation, processed_dir, classes,
            stats, device=device,
        )
        probabilities[model_id][validation.window_row.to_numpy(int)] = proba
        scores = participant_metrics(
            validation.reset_index(drop=True), proba, classes,
            model_id=model_id, fold=fold, seed=SEED,
        )
        for key, value in info.items():
            if key not in ("seed", "model_id"):
                scores[key] = value
        history["validation_fold"] = fold
        model_scores.append(scores)
        model_histories.append(history)
        model_predictions.append(pd.DataFrame({
            "window_key": validation.window_key.to_numpy(),
            "subject_id": validation.subject_id.to_numpy(),
            "validation_fold": fold,
            "model_id": model_id,
            "true_label": validation.label.to_numpy(int),
            "predicted_label": predictions_from_probabilities(proba, classes),
        }))
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Overwrite this model's current outputs after every finished fold.
        scores_so_far = pd.concat(model_scores, ignore_index=True)
        predictions_so_far = pd.concat(model_predictions, ignore_index=True)
        history_so_far = pd.concat(model_histories, ignore_index=True)
        scores_so_far.to_csv(
            output_dir / f"03_{model_id}_metrics.csv", index=False,
        )
        predictions_so_far.to_csv(
            output_dir / f"03_{model_id}_oof_predictions.csv", index=False,
        )
        history_so_far.to_csv(
            output_dir / f"03_{model_id}_learning_history.csv", index=False,
        )
        np.savez_compressed(
            output_dir / f"03_{model_id}_oof_probabilities.npz",
            window_key=dev.window_key.to_numpy(dtype=str),
            class_labels=classes,
            validation_fold=dev.validation_fold.to_numpy(int),
            scores=probabilities[model_id][development_rows],
            seed=np.asarray(SEED),
            epochs=np.asarray(NEURAL_EPOCHS),
            patience=np.asarray(NEURAL_PATIENCE),
            batch_size=np.asarray(NEURAL_BATCH_SIZE),
        )
        print(
            f"Saved fold {fold} | participant-mean macro F1 "
            f"{scores.macro_f1.mean():.4f} | fit {info['fit_seconds']:.1f} s",
            flush=True,
        )

    completed_scores = pd.concat(model_scores, ignore_index=True)
    completed_history = pd.concat(model_histories, ignore_index=True)
    fold_summary = completed_scores.groupby(
        "validation_fold", as_index=False,
    ).agg(
        participants=("subject_id", "nunique"),
        participant_mean_macro_f1=("macro_f1", "mean"),
        fit_seconds=("fit_seconds", "max"),
    )
    print(
        f"\n{model_id} | equal-participant Development macro F1: "
        f"{completed_scores.macro_f1.mean():.4f}",
        flush=True,
    )
    display(fold_summary)

    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    for fold, curve in completed_history.groupby("validation_fold", sort=True):
        ax.plot(
            curve.epoch, curve.inner_validation_macro_f1,
            marker="o", label=f"Fold {fold}",
        )
    ax.set(
        xlabel="Training epoch",
        ylabel="Inner Training-only validation macro F1",
        title=f"{model_id}: early-stopping evidence",
    )
    ax.legend(ncol=min(len(common_folds), 5), fontsize=9)
    ax.grid(alpha=0.2)
    save_figure(fig, output_dir / f"03_{model_id}_learning_curve.png")
    plt.show()
    return fold_summary

**06. sEMG-Only 1D-CNN**

The D1 sEMG network learns temporal patterns from the 12 surface-electromyography channels. Three convolutional stages progressively reduce the 300-ms source-grid sequence before a classification head. This cell trains the model across all five identical participant-disjoint Development folds, displays epoch-level optimisation, reports each fold's participant-mean macro F1 and stores its own outputs. Acceleration is excluded from the predictor.

In [ ]:
# =========================== 06. sEMG-Only 1D-CNN ============================

# ---------------------- 1. Participant-Grouped Model Fit ---------------------
D1_EMG_CNN_FOLD_SUMMARY = run_unimodal_arm("D1_emg_cnn")

**07. Acceleration-Only 1D-CNN**

The matched D1 convolutional network receives 36 acceleration axes from the 12 Delsys modules, without sEMG. It uses the same source-aligned windows, class definitions, folds and training budget as the sEMG CNN. The model's epoch logs, per-fold scores, out-of-fold predictions and learning curve are exported separately.

In [ ]:
# ======================= 07. Acceleration-Only 1D-CNN ========================

# ---------------------- 1. Participant-Grouped Model Fit ---------------------
D1_ACC_CNN_FOLD_SUMMARY = run_unimodal_arm("D1_acc_cnn")

**08. sEMG-Only TCN**

This temporal convolutional network uses residual dilated convolutions on the 12-channel sEMG sequence. Its temporal receptive field differs from the compact D1 CNN, allowing a controlled comparison of unimodal temporal encoders. Training-only normalisation and early stopping remain inside each outer Training partition.

In [ ]:
# ============================= 08. sEMG-Only TCN =============================

# ---------------------- 1. Participant-Grouped Model Fit ---------------------
D2_EMG_TCN_FOLD_SUMMARY = run_unimodal_arm("D2_emg_tcn")

**09. Acceleration-Only TCN**

The acceleration TCN processes the 36 paired acceleration columns using the same residual architecture family as the sEMG TCN. Interpolated acceleration rows do not represent independent 2,000-Hz measurements. The cell reports participant-level Development performance and model-specific fitting times without using Test participants.

In [ ]:
# ========================= 09. Acceleration-Only TCN =========================

# ---------------------- 1. Participant-Grouped Model Fit ---------------------
D2_ACC_TCN_FOLD_SUMMARY = run_unimodal_arm("D2_acc_tcn")

**10. sEMG-Only BiGRU**

The D3 bidirectional gated recurrent unit (BiGRU) models temporal dependencies in the sEMG window after time-axis pooling. Bidirectionality uses the complete offline window and does not establish causal or real-time operation. Its own fold results and training curves are displayed and saved independently.

In [ ]:
# ============================ 10. sEMG-Only BiGRU ============================

# ---------------------- 1. Participant-Grouped Model Fit ---------------------
D3_EMG_BIGRU_FOLD_SUMMARY = run_unimodal_arm("D3_emg_bigru")

**11. Acceleration-Only BiGRU**

The final unimodal arm applies the same BiGRU family to the acceleration-only sequence. Together, the six individually executed architectures provide a matched Development comparison of modality and temporal representation. After this cell, the consolidated outputs can be assembled for Notebook 04.

In [ ]:
# ======================== 11. Acceleration-Only BiGRU ========================

# ---------------------- 1. Participant-Grouped Model Fit ---------------------
D3_ACC_BIGRU_FOLD_SUMMARY = run_unimodal_arm("D3_acc_bigru")

**12. Consolidate the Six Unimodal Neural Experiments**

Read the named current outputs from the six model cells, check the source window keys, fold assignments, class order and training settings, and construct the combined neural tables. Each model retains its own metric, history, prediction and probability files. Incomplete model runs cannot enter the aggregate comparison.

In [ ]:
# ================ 12. Consolidate Unimodal Neural Experiments ================

neural_scores, neural_curves, neural_oof = [], [], []
development_rows = dev.window_row.to_numpy(int)
for model_id in UNIMODAL_MODELS:
    prefix = output_dir / f"03_{model_id}"
    score_path = output_dir / f"03_{model_id}_metrics.csv"
    pred_path = output_dir / f"03_{model_id}_oof_predictions.csv"
    hist_path = output_dir / f"03_{model_id}_learning_history.csv"
    probability_path = output_dir / f"03_{model_id}_oof_probabilities.npz"
    required = (score_path, pred_path, hist_path, probability_path)
    missing = [p.name for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            f"Complete the {model_id} cell first; missing: {missing}"
        )
    with np.load(probability_path, allow_pickle=False) as saved:
        if not np.array_equal(saved["window_key"], dev.window_key.to_numpy(dtype=str)):
            raise ValueError(f"{model_id}: source windows changed; rerun this model.")
        if not np.array_equal(saved["class_labels"], classes):
            raise ValueError(f"{model_id}: class order changed; rerun this model.")
        if not np.array_equal(saved["validation_fold"],
                              dev.validation_fold.to_numpy(int)):
            raise ValueError(f"{model_id}: fold assignment changed; rerun this model.")
        settings_match = (
            int(saved["seed"]) == SEED
            and int(saved["epochs"]) == NEURAL_EPOCHS
            and int(saved["patience"]) == NEURAL_PATIENCE
            and int(saved["batch_size"]) == NEURAL_BATCH_SIZE
        )
        if not settings_match:
            raise ValueError(f"{model_id}: training settings changed; rerun this model.")
        proba = saved["scores"]
        if proba.shape != (len(dev), len(classes)):
            raise ValueError(f"{model_id}: incomplete probability array.")
        if not np.isfinite(proba).all():
            raise ValueError(f"{model_id}: unfinished fold; complete this model's cell.")
        if not np.allclose(proba.sum(axis=1), 1.0, atol=1e-4):
            raise ValueError(f"{model_id}: invalid probability rows.")
        probabilities[model_id][development_rows] = proba

    scores = pd.read_csv(score_path)
    predictions = pd.read_csv(pred_path)
    history = pd.read_csv(hist_path)
    if set(scores.validation_fold.astype(int)) != set(common_folds):
        raise ValueError(f"{model_id}: incomplete Development folds.")
    if (
        len(predictions) != len(dev)
        or not predictions.window_key.is_unique
        or set(predictions.window_key) != set(dev.window_key)
    ):
        raise ValueError(f"{model_id}: incomplete/misaligned prediction rows.")
    if set(scores.subject_id.astype(str)) != set(dev.subject_id.astype(str)):
        raise ValueError(f"{model_id}: participant-level score coverage is incomplete.")
    neural_scores.append(scores)
    neural_oof.append(predictions)
    neural_curves.append(history)
    print(
        f"Loaded {model_id}: {scores.subject_id.nunique()} people | "
        f"macro F1 {scores.macro_f1.mean():.4f}"
    )

neural_metrics = pd.concat(neural_scores, ignore_index=True)
neural_history = pd.concat(neural_curves, ignore_index=True)
neural_predictions = pd.concat(neural_oof, ignore_index=True)
neural_metrics.to_csv(output_dir / "03_dev_neural_metrics.csv", index=False)
neural_history.to_csv(output_dir / "03_neural_learning_history.csv", index=False)
neural_predictions.to_csv(output_dir / "03_neural_oof_predictions.csv", index=False)
display(
    neural_metrics.groupby("model_id", as_index=False)
    .macro_f1.mean().sort_values("macro_f1", ascending=False)
)

**13. Save Matched Out-of-Fold Probabilities and Select the Comparator**

The original `window_row` determines probability order for every model; Test rows remain unfilled. Combining classical and neural Development results produces the same output names consumed by Notebook 04. The strongest unimodal comparator is chosen on participant-balanced Development macro F1 only.

In [ ]:
# =============== 13. Matched Probabilities and Model Selection ===============

for model_id, proba in probabilities.items():
    validation_proba = proba[dev.window_row.to_numpy(int)]
    if (
        not np.isfinite(validation_proba).all()
        or not np.allclose(validation_proba.sum(axis=1), 1.0, atol=1e-4)
    ):
        raise ValueError(f"Missing/invalid Development probabilities: {model_id}")

np.savez_compressed(output_dir / "03_oof_probabilities.npz", **probabilities)
all_metrics = pd.concat([classical_metrics, neural_metrics], ignore_index=True)
all_metrics.to_csv(output_dir / "03_all_unimodal_metrics.csv", index=False)
chosen, summary = select_model(all_metrics, list(probabilities))
summary.to_csv(output_dir / "03_unimodal_model_summary.csv", index=False)
print("Development-selected strongest unimodal model:", chosen)
display(summary)

**14. Define Multi-Seed Robustness for Development Finalists**

The primary matched model comparison uses seed 42. To characterise training variability without treating random seeds as additional participants, select the leading Development neural architecture in each modality and repeat its participant-grouped fits with seeds 43 and 44. The sEMG and acceleration finalists run in separate cells.

In [ ]:
# ==================== 14. Define Finalist Seed Evaluation ====================

RUN_FINALIST_ROBUSTNESS = True
EXTRA_SEEDS = (43, 44)
emg_finalist = select_model(
    neural_metrics, [x for x in UNIMODAL_MODELS if "_emg_" in x]
)[0]
acc_finalist = select_model(
    neural_metrics, [x for x in UNIMODAL_MODELS if "_acc_" in x]
)[0]
print("sEMG finalist:", emg_finalist, "| acceleration finalist:", acc_finalist)

def evaluate_finalist_seeds(model_id):
    """Evaluate two additional random seeds for one Development finalist."""
    arm_scores, arm_histories = [], []
    for replicate_seed in EXTRA_SEEDS:
        for fold in common_folds:
            fitting = dev.loc[dev.validation_fold != fold]
            validation = dev.loc[dev.validation_fold == fold]
            print(
                f"\n{model_id} | seed {replicate_seed} | fold {fold}",
                flush=True,
            )
            model, stats, history, info = fit_neural(
                model_id, fitting, processed_dir, classes,
                device=device, seed=replicate_seed,
                epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
                batch_size=NEURAL_BATCH_SIZE, show_progress=True,
            )
            proba = predict_neural(
                model_id, model, validation, processed_dir,
                classes, stats, device=device,
            )
            scores = participant_metrics(
                validation.reset_index(drop=True), proba, classes,
                model_id=model_id, fold=fold, seed=replicate_seed,
            )
            scores["fit_seconds"] = info["fit_seconds"]
            arm_scores.append(scores)
            history["validation_fold"] = fold
            arm_histories.append(history)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            pd.concat(arm_scores, ignore_index=True).to_csv(
                output_dir / f"03_{model_id}_extra_seed_metrics.csv",
                index=False,
            )
            pd.concat(arm_histories, ignore_index=True).to_csv(
                output_dir / f"03_{model_id}_extra_seed_history.csv",
                index=False,
            )
            print(
                f"Saved seed {replicate_seed}, fold {fold}: "
                f"mean participant macro F1 {scores.macro_f1.mean():.4f}",
                flush=True,
            )
    result = pd.concat(arm_scores, ignore_index=True)
    display(result.groupby("seed", as_index=False).macro_f1.mean())
    return result

**15. Additional Seeds: Selected sEMG Model**

Train only the Development-selected sEMG neural finalist with two additional random seeds. Each fold prints epoch progress and writes model-specific robustness metrics and learning histories. These are secondary stability results, not a replacement for the primary seed-42 model selection.

In [ ]:
# ================= 15. Selected sEMG Model Additional Seeds ==================

if RUN_FINALIST_ROBUSTNESS:
    emg_seed_metrics = evaluate_finalist_seeds(emg_finalist)
else:
    print("Additional-seed robustness was not requested for this run.")

**16. Additional Seeds: Selected Acceleration Model**

Apply the same seed and participant-fold protocol to the strongest acceleration-only neural model. The combined robustness files preserve separate model/seed identifiers so that variability is not mistaken for independent participant evidence.

In [ ]:
# ============= 16. Selected Acceleration Model Additional Seeds ==============

if RUN_FINALIST_ROBUSTNESS:
    acceleration_seed_metrics = evaluate_finalist_seeds(acc_finalist)
    extra_metrics = pd.concat(
        [
            pd.read_csv(output_dir / f"03_{model_id}_extra_seed_metrics.csv")
            for model_id in (emg_finalist, acc_finalist)
        ],
        ignore_index=True,
    )
    extra_histories = pd.concat(
        [
            pd.read_csv(output_dir / f"03_{model_id}_extra_seed_history.csv")
            for model_id in (emg_finalist, acc_finalist)
        ],
        ignore_index=True,
    )
    extra_metrics.to_csv(
        output_dir / "03_neural_multiseed_metrics.csv", index=False,
    )
    extra_histories.to_csv(
        output_dir / "03_neural_multiseed_history.csv", index=False,
    )
    display(extra_metrics.groupby(["model_id", "seed"], as_index=False)
            .macro_f1.mean())

**17. Compare Unimodal Errors and Learning Behaviour**

Summarise the common Development participant-macro F1 comparison, illustrate aggregate training-loss behaviour and show the actual out-of-fold confusion matrix for the selected unimodal model. The individual architecture cells above also provide model-specific inner-validation curves.

In [ ]:
# =============== 17. Development Comparison and Error Figures ================

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
ordering = summary.model_id.tolist()
axes[0].barh(ordering[::-1], summary.mean_subject_macro_f1.to_numpy()[::-1])
axes[0].set(
    xlabel="Development participant-mean macro F1",
    title="Matched unimodal comparison",
)
curves = neural_history.groupby("epoch", as_index=False).agg(
    train_loss=("train_loss", "mean"),
)
axes[1].plot(curves.epoch, curves.train_loss, marker="o")
axes[1].set(
    xlabel="Training epoch", ylabel="Mean training loss",
    title="Neural optimisation across completed folds",
)
for ax in axes:
    ax.grid(alpha=0.2)
save_figure(fig, output_dir / "03_learning_curves.png")
plt.show()

selected_rows = dev.sort_values("window_row")
selected_proba = probabilities[chosen][selected_rows.window_row.to_numpy(int)]
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    selected_rows.label.to_numpy(int),
    predictions_from_probabilities(selected_proba, classes),
    labels=classes,
)
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
im = ax.imshow(cm, aspect="auto")
ax.set(
    xlabel="Predicted source label", ylabel="True source label",
    title=f"Development out-of-fold errors: {chosen}",
)
ax.set_xticks(np.arange(len(classes)), labels=classes, rotation=90)
ax.set_yticks(np.arange(len(classes)), labels=classes)
fig.colorbar(im, ax=ax, label="Window count")
save_figure(fig, output_dir / "03_class_errors.png")
plt.show()

**18. Research Outputs and Next Stage**

Review the model-specific and consolidated Development evidence, together with the fold assignments and source window keys. Notebook 04 consumes the current aggregate probabilities and summary tables to compare fusion against the Development-selected unimodal comparator; no Test observations were used in the selection.

In [ ]:
# ====================== 18. Stage Outputs and Download =======================

display(pd.DataFrame({
    "output_file": [
        path.name for path in sorted(output_dir.glob("03_*"))
        if path.is_file()
    ]
}))
download_outputs_from_colab(
    folder="multimodal_movement_learning/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)